In [15]:
import json
import re
from pathlib import Path
import pandas as pd

In [16]:
base_dir = Path(
    r"e:\King's College London\Department of Psychosis Shared Database Initiative - Documents\Read App"
)

participants_path = base_dir / "participants_info.csv"
modalities_path = base_dir / "modalities_info.csv"
descriptor_path = base_dir / "../Storage Repository/variable_descriptor.json"

participants = pd.read_csv(participants_path)
modalities = pd.read_csv(modalities_path)

def load_group_labels(path: Path) -> dict[str, str]:
    text = path.read_text(encoding="utf-8")

    try:
        descriptor = json.loads(text)
        values = descriptor.get("group", {}).get("values", {})
        labels = {str(key): str(value) for key, value in values.items()}

    except json.JSONDecodeError:
        labels = {}

        group_match = re.search(
            r'"group"\s*:\s*\{.*?"values"\s*:\s*\{(.*?)\}\s*\}',
            text,
            flags=re.S
        )

        if group_match:
            labels.update(
                dict(re.findall(r'"([^"]+)"\s*:\s*"([^"]+)"', group_match.group(1)))
            )

    labels.update({
        "0": "HC",
        "0.1": "HC Sibling",
        "1": "SCHZ",
        "1.1": "SCHZ Sibling",
        "2": "BD",
        "3": "FEP",
        "4": "MDD",
        "5": "CHR",
        "6": "ADHD",
        "7": "PTSD",
        "8": "GAD: general anxiety disorder",
        "9": "SAD: social anxiety disorder",
        "10": "SUD: substance use disorder",
        "11": "OCD: obsessive-compulsive disorder",
        "12": "SEAD: separation anxiety disorder",
        "13": "ASD: autistic spectrum disorder",
        "14": "CPS: chronic pain syndrome",
        "15": "Stroke",
        "16": "BPD: Borderline personality disorder",
        "17": "ARMS: at risk mental state",
        "18": "AN: Anorexia nerviosa",
        "19": "WR: Weight-recovered anorexia nerviosa",
        "20": "AUD: Alcohol use disorder",
        "21": "Panic Episode",
        "22": "Agoraphobia"
    })

    return labels

group_labels = load_group_labels(descriptor_path)

diagnosis_order = [
    "CHR", "ARMS", "SCHZ", "SCHZ Sibling", "BD", "FEP", "MDD", "ADHD", "PTSD",
    "GAD", "SAD", "Panic Episode", "Agoraphobia", "OCD", "ASD", "BPD",
    "SUD", "AUD", "SEAD", "CPS", "Stroke", "AN", "WR",
    "HC Sibling", "HC", "Other"
]

def normalize_group_value(value):
    if pd.isna(value):
        return None

    value_str = str(value).strip()

    if value_str == "":
        return None

    try:
        numeric_value = float(value_str)
        if numeric_value.is_integer():
            return str(int(numeric_value))
        return str(numeric_value)

    except (TypeError, ValueError):
        return value_str

def diagnosis_label(value):
    key = normalize_group_value(value)

    if key is None:
        return "Other"

    return group_labels.get(key, key)

def compact_label(label):
    if label is None:
        return "Other"

    label = str(label).strip()

    if label == "":
        return "Other"

    return label.split(":", 1)[0].strip()

def split_group_values(value):
    if pd.isna(value):
        return ["Other"]

    value_str = str(value).strip()

    if value_str == "":
        return ["Other"]

    return [
        part.strip()
        for part in value_str.split(";")
        if part.strip() != ""
    ]

def format_diagnoses(series):
    all_labels = []

    for value in series:
        group_values = split_group_values(value)

        for group_value in group_values:
            label = compact_label(diagnosis_label(group_value))
            all_labels.append(label)

    if not all_labels:
        all_labels = ["Other"]

    counts = pd.Series(all_labels).value_counts()

    ordered_labels = [label for label in diagnosis_order if label in counts.index]
    ordered_labels.extend([label for label in counts.index if label not in ordered_labels])

    return ", ".join(
        f"{label} ({int(counts[label])})"
        for label in ordered_labels
    )

def format_modalities(df):
    parts = []

    if "anat" in df.columns and df["anat"].max() == 1:
        parts.append("sMRI (T1w)")

    if "fmri" in df.columns and df["fmri"].max() == 1:
        parts.append("fMRI")

    if "dwi" in df.columns and df["dwi"].max() == 1:
        parts.append("dMRI")

    return ", ".join(parts) if parts else "Unknown"

participant_summary = (
    participants.groupby("dataset", as_index=False)
    .agg(
        sample_size=("participant_id", "nunique"),
        diagnoses=("group", format_diagnoses),
    )
)

modality_summary = (
    modalities.groupby("dataset")
    .apply(format_modalities)
    .rename("modalities")
    .reset_index()
)

summary = (
    participant_summary
    .merge(modality_summary, on="dataset", how="outer")
    .sort_values("dataset")
    .reset_index(drop=True)
)

summary["modalities"] = summary["modalities"].fillna("Unknown")
summary["diagnoses"] = summary["diagnoses"].fillna("Other")

summary = summary[["dataset", "sample_size", "modalities", "diagnoses"]]

summary.to_csv("summary.csv", index=False)

summary


C:\Users\Sergio\AppData\Local\Temp\ipykernel_5212\4061897715.py:172: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(format_modalities)


,dataset,sample_size,modalities,diagnoses
0,ACHRP,47,"sMRI (T1w), fMRI","CHR (19), HC (28)"
1,AGERISK,189,"sMRI (T1w), fMRI",HC (189)
2,AOMICID,928,"sMRI (T1w), fMRI, dMRI",HC (928)
3,AOMICPIOP,216,"sMRI (T1w), fMRI, dMRI",HC (216)
4,AOMICPIOPII,226,"sMRI (T1w), fMRI, dMRI",HC (226)
5,ARTS,101,"sMRI (T1w), fMRI, dMRI","SCHZ (96), HC (5)"
6,BCSPS,71,"sMRI (T1w), fMRI","SCHZ (46), HC (25)"
7,BEACON,183,sMRI (T1w),"AN (58), WR (58), HC (67)"
8,BGSCHZ,250,"sMRI (T1w), fMRI, dMRI","SCHZ (136), HC (114)"
9,CANDI,103,sMRI (T1w),"SCHZ (20), BD (54), HC (29)"
